# Card Database and Starter-Deck EDA

**Purpose.** Audit the official English card catalogue and starter deck
before changing either policy or deck construction.

**Decision question.** Which card-pool and deck properties constrain our
first agent experiments?

This is a simulation competition: EDA means catalogue, deck, state-space,
and episode analysis - not train/test target exploration. Run with the
`pokemon-tcg-ai-battle` competition data attached. A local `data/raw`
fallback is supported.

## 1. Configuration

The input resolver avoids account-specific paths. We keep plots consistent
and separate card identity from move-level records because one card can
occupy several catalogue rows.

In [ ]:
from collections import Counter
from math import comb
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA_FILENAME = "EN_Card_Data.csv"
sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 50)

def find_file(filename: str) -> Path:
    root = Path("/kaggle/input")
    matches = sorted(root.rglob(filename)) if root.exists() else []
    if matches:
        return matches[0]
    local = Path("../data/raw") / filename
    if local.exists():
        return local
    raise FileNotFoundError(
        f"Attach competition data or place {filename} in data/raw/."
    )

card_path = find_file(DATA_FILENAME)
print(f"Catalogue: {card_path}")

## 2. Schema and representation audit

We distinguish catalogue rows, unique card IDs, and card-move records.
Structural missingness is expected: Energy and Trainer cards do not have
Pokemon HP or evolution fields.

In [ ]:
raw = pd.read_csv(card_path)
cards = raw.rename(columns=lambda x: re.sub(
    r"[^a-z0-9]+", "_", x.strip().lower()
).strip("_"))

summary = pd.Series({
    "catalogue_rows": len(cards),
    "columns": cards.shape[1],
    "unique_card_ids": cards.card_id.nunique(),
    "exact_duplicate_rows": cards.duplicated().sum(),
})
display(summary.to_frame("value"))
display(cards.head())

quality = pd.DataFrame({
    "dtype": cards.dtypes.astype(str),
    "missing": cards.isna().sum(),
    "missing_pct": cards.isna().mean().mul(100).round(2),
    "unique": cards.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(quality)

**Interpretation.** Do not globally impute the catalogue. Parse and analyze
fields within relevant card categories, and use `Card ID` as the stable
identity key. Names and move rows are not unique enough for policy logic.

## 3. Card-pool composition

Collapse identity fields to one record per card for composition plots.
Attacks remain a separate one-to-many table for later action scoring.

In [ ]:
category_col = "stage_pok_mon_type_energy_and_trainer"
identity = [
    "card_id", "card_name", "expansion", category_col, "rule", "category",
    "previous_stage", "hp", "type", "weakness", "resistance_type", "retreat",
]
identity = [column for column in identity if column in cards]
unique_cards = cards[identity].groupby("card_id", as_index=False).first()

counts = unique_cards[category_col].fillna("Missing").value_counts().head(15)
ax = counts.sort_values().plot.barh(
    figsize=(10, 6), color=sns.color_palette("viridis", len(counts))
)
ax.set(title="Largest card categories", xlabel="Unique cards", ylabel="Category")
plt.tight_layout()
plt.show()

unique_cards["hp_numeric"] = pd.to_numeric(unique_cards.hp, errors="coerce")
unique_cards["retreat_numeric"] = pd.to_numeric(unique_cards.retreat, errors="coerce")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(unique_cards.hp_numeric.dropna(), bins=30, ax=axes[0])
axes[0].set_title("Printed HP distribution")
sns.countplot(x=unique_cards.retreat_numeric, ax=axes[1], color="#2a788e")
axes[1].set_title("Retreat-cost distribution")
plt.tight_layout()
plt.show()

**Interpretation.** Catalogue frequency is descriptive, not a deck recipe.
Deck strength depends on legal counts, evolution support, energy curve,
interactions, and whether the policy can sequence them correctly.

## 4. Starter-deck audit

Basic Energy can exceed the ordinary four-copy heuristic. We flag other
large counts for review rather than claiming that a simplified check is the
official legality test. The simulator start result remains authoritative.

In [ ]:
def find_deck() -> Path:
    candidates = [Path("../agent/deck.csv"), Path("agent/deck.csv")]
    candidates += sorted(Path("/kaggle/input").rglob("sample_submission/deck.csv"))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find a repository or official starter deck.")

deck_path = find_deck()
deck = [int(x) for x in deck_path.read_text().splitlines() if x.strip()]
deck_counts = pd.Series(Counter(deck), name="copies").rename_axis("card_id").reset_index()
columns = ["card_id", "card_name", category_col, "hp", "retreat"]
deck_view = deck_counts.merge(unique_cards[columns], on="card_id", how="left")
deck_view = deck_view.sort_values(["copies", "card_id"], ascending=[False, True])

print(f"Deck: {deck_path}")
print(f"Cards: {len(deck)}; unique IDs: {len(deck_counts)}")
display(deck_view)
assert len(deck) == 60, "Submission decks require exactly 60 cards."
assert deck_view.card_name.notna().all(), "All deck IDs must exist in the catalogue."

basic_energy = deck_view[category_col].fillna("").str.contains(
    "Basic Energy", case=False, regex=False
)
print("Non-Basic-Energy entries above four copies (manual review):")
display(deck_view[(deck_view.copies > 4) & ~basic_energy])

## 5. Deck roles and setup consistency

A legal 60-card list can still be operationally fragile. The opening hand
needs at least one Basic Pokemon, so we compute the exact hypergeometric
setup probability and compare it with nearby Basic-Pokemon counts. This is
the most actionable deck-construction diagnostic in this EDA.

In [ ]:
def deck_role(value: object) -> str:
    label = str(value)
    if "Basic Energy" in label:
        return "Basic Energy"
    if label.startswith("Basic Pok"):
        return "Basic Pokemon"
    if "Pok" in label:
        return "Evolution Pokemon"
    for role in ("Supporter", "Item", "Tool", "Stadium", "Special Energy"):
        if role in label:
            return role
    return "Other"

deck_view["role"] = deck_view[category_col].map(deck_role)
role_counts = deck_view.groupby("role", as_index=False).copies.sum()
role_counts["share_pct"] = role_counts.copies.div(len(deck)).mul(100).round(1)
display(role_counts.sort_values("copies", ascending=False))

ax = role_counts.sort_values("copies").plot.barh(
    x="role", y="copies", legend=False, figsize=(9, 5), color="#2a788e"
)
ax.set(title="Starter-deck composition by functional role", xlabel="Cards", ylabel="")
plt.tight_layout()
plt.show()

OPENING_HAND_SIZE = 7
current_basic_count = int(
    deck_view.loc[deck_view.role.eq("Basic Pokemon"), "copies"].sum()
)

def setup_probability(basic_count: int, deck_size: int = 60) -> float:
    if basic_count <= 0:
        return 0.0
    return 1 - comb(deck_size - basic_count, OPENING_HAND_SIZE) / comb(
        deck_size, OPENING_HAND_SIZE
    )

setup_curve = pd.DataFrame({"basic_pokemon": np.arange(4, 17)})
setup_curve["setup_probability"] = setup_curve.basic_pokemon.map(setup_probability)
current_setup_probability = setup_probability(current_basic_count)
current_mulligan_probability = 1 - current_setup_probability

display(pd.Series({
    "basic_pokemon": current_basic_count,
    "opening_hand_size": OPENING_HAND_SIZE,
    "p_at_least_one_basic": current_setup_probability,
    "p_no_basic_mulligan": current_mulligan_probability,
}).to_frame("value"))

ax = setup_curve.plot(
    x="basic_pokemon", y="setup_probability", marker="o", legend=False,
    figsize=(9, 4), color="#2a788e"
)
ax.scatter([current_basic_count], [current_setup_probability], color="#d1495b", zorder=3)
ax.axhline(0.80, color="grey", linestyle="--", linewidth=1, label="80% reference")
ax.set(
    title="Probability that a seven-card opening hand contains a Basic Pokemon",
    xlabel="Basic Pokemon in 60-card deck", ylabel="Probability", ylim=(0, 1),
)
ax.legend()
plt.tight_layout()
plt.show()

**Decision.** With only six Basic Pokemon, the starter deck has substantial
mulligan risk. Policy improvements cannot fully repair hands that fail to
set up; future deck experiments should vary Basic-Pokemon count separately
from policy experiments so their effects remain attributable.

## 6. Evolution-line support

Evolution cards are only actionable when their printed previous stage is
also present. We audit every evolution line by name and weight the result by
deck copies. This catches dead evolution cards without pretending to model
draw order or board state.

In [ ]:
deck_names = set(deck_view.card_name.dropna().astype(str))
evolution_columns = [
    "card_id", "card_name", "previous_stage", "copies", "role"
]
evolution_view = deck_counts.merge(
    unique_cards[["card_id", "card_name", "previous_stage"]],
    on="card_id", how="left"
)
evolution_view = evolution_view[evolution_view.previous_stage.notna()].copy()
evolution_view["previous_stage_in_deck"] = evolution_view.previous_stage.isin(deck_names)
evolution_view["support_copies"] = evolution_view.previous_stage.map(
    deck_view.set_index("card_name").copies
).fillna(0).astype(int)
display(evolution_view[evolution_columns[:-1] + ["previous_stage_in_deck", "support_copies"]])

evolution_copies = int(evolution_view.copies.sum())
supported_evolution_copies = int(
    evolution_view.loc[evolution_view.previous_stage_in_deck, "copies"].sum()
)
evolution_support_rate = (
    supported_evolution_copies / evolution_copies if evolution_copies else 1.0
)
print(f"Evolution support rate by copies: {evolution_support_rate:.1%}")

**Decision.** The current Stage 1 line is structurally supported. The agent
should therefore preserve evolution opportunities and favor development
actions early, while episode telemetry determines whether those cards are
drawn and sequenced reliably in practice.

## 7. Move structure and attack efficiency

The first Kaggle run confirmed a 60-card starter deck with only nine unique
IDs. That concentration makes aggregate catalogue plots insufficient: we
need to inspect the attacks, energy requirements, evolution support, and
retreat burden of cards the baseline can actually draw.

In [ ]:
move_columns = [
    "card_id", "card_name", "move_name", "cost", "damage",
    "effect_explanation",
]
moves = cards[[column for column in move_columns if column in cards]].copy()
moves = moves[moves.move_name.notna()].copy()
moves["printed_damage"] = pd.to_numeric(moves.damage, errors="coerce")
moves["damage_floor"] = pd.to_numeric(
    moves.damage.astype(str).str.extract(r"(\d+)")[0], errors="coerce"
)
moves["energy_symbols"] = moves.cost.fillna("").str.count(r"\{[^}]+\}")
moves["variable_damage"] = moves.damage.notna() & moves.printed_damage.isna()
moves["damage_per_energy"] = moves.damage_floor.div(
    moves.energy_symbols.replace(0, np.nan)
)

deck_moves = moves[moves.card_id.isin(deck_counts.card_id)].merge(
    deck_counts, on="card_id", how="left"
).sort_values(["copies", "card_id", "move_name"], ascending=[False, True, True])
display(deck_moves[
    ["card_id", "card_name", "copies", "move_name", "cost", "damage",
     "energy_symbols", "damage_per_energy", "variable_damage"]
])

plotted_moves = deck_moves.dropna(subset=["damage_floor", "energy_symbols"]).copy()
if not plotted_moves.empty:
    ax = sns.scatterplot(
        data=plotted_moves, x="energy_symbols", y="damage_floor",
        hue="card_name", size="copies", sizes=(70, 220), s=120
    )
    ax.set(
        title="Printed damage floor versus attack energy requirement",
        xlabel="Printed energy symbols", ylabel="Damage floor"
    )
    plt.tight_layout()
    plt.show()

deck_detail = deck_view.copy()
deck_detail["weighted_retreat"] = (
    pd.to_numeric(deck_detail.retreat, errors="coerce").fillna(0)
    * deck_detail.copies
)
deck_summary = pd.Series({
    "cards": len(deck),
    "unique_ids": len(deck_counts),
    "basic_energy_cards": int(deck_view.loc[basic_energy, "copies"].sum()),
    "non_energy_cards": int(deck_view.loc[~basic_energy, "copies"].sum()),
    "deck_moves": len(deck_moves),
    "variable_damage_moves": int(deck_moves.variable_damage.sum()),
    "weighted_retreat_cost": float(deck_detail.weighted_retreat.sum()),
    "basic_pokemon": current_basic_count,
    "opening_setup_probability": current_setup_probability,
    "opening_mulligan_probability": current_mulligan_probability,
    "evolution_support_rate": evolution_support_rate,
})
display(deck_summary.to_frame("value"))

In [ ]:
import json

if Path("/kaggle/working").exists():
    output = Path("/kaggle/working")
else:
    output = Path("../scratch") if Path.cwd().name == "notebooks" else Path("scratch")
    output.mkdir(parents=True, exist_ok=True)
eda_summary = {
    "catalogue_rows": int(len(cards)),
    "unique_card_ids": int(cards.card_id.nunique()),
    "deck": {key: float(value) for key, value in deck_summary.items()},
    "deck_card_ids": {str(int(row.card_id)): int(row.copies)
                      for row in deck_counts.itertuples()},
}
(output / "card_eda_summary.json").write_text(
    json.dumps(eda_summary, indent=2)
)
print(f"Saved summary to {output / 'card_eda_summary.json'}")

## 8. Card-reference PDF audit

The official English PDF is a visual lookup from simulator card ID to card
name, expansion, collection number, and card image. The CSV remains the
computational source of truth; bulk OCR would be slower, noisier, and would
unnecessarily reproduce protected card content.

A repository audit of the 137.7 MB English document found 1,306 pages and
rendered pages 1, 2, 654, 1,305, and 1,306 for structural sampling. Set
`RUN_PDF_RENDER = True` only when a human needs to resolve a visual ambiguity.

In [ ]:
RUN_PDF_RENDER = False
PDF_SAMPLE_PAGES = (1, 2, 654, 1305, 1306)
pdf_candidates = sorted(Path("/kaggle/input").rglob("*List_EN.pdf"))
if not pdf_candidates:
    pdf_candidates = sorted(Path("../tmp/pdfs").glob("*.pdf"))
pdf_path = pdf_candidates[0] if pdf_candidates else None

pdf_audit = {
    "role": "visual card-ID reference; CSV is the analysis source",
    "observed_pages": 1306,
    "sampled_pages": list(PDF_SAMPLE_PAGES),
    "file_found": pdf_path is not None,
    "size_mb": round(pdf_path.stat().st_size / 1_000_000, 2) if pdf_path else None,
}
display(pd.Series(pdf_audit).to_frame("value"))

if RUN_PDF_RENDER:
    import fitz
    from IPython.display import display as display_image
    from PIL import Image

    if pdf_path is None:
        raise FileNotFoundError("Attach the English Card ID PDF before rendering.")
    document = fitz.open(pdf_path)
    assert document.page_count == pdf_audit["observed_pages"]
    for page_number in PDF_SAMPLE_PAGES:
        page = document[page_number - 1]
        pixmap = page.get_pixmap(matrix=fitz.Matrix(0.75, 0.75), alpha=False)
        image = Image.frombytes("RGB", [pixmap.width, pixmap.height], pixmap.samples)
        print(f"Reference page {page_number}")
        display_image(image)
else:
    print("PDF rendering skipped by default; enable only for targeted visual review.")

eda_summary["pdf_reference"] = pdf_audit
(output / "card_eda_summary.json").write_text(json.dumps(eda_summary, indent=2))

## 9. Decision summary and next experiment

The EDA now answers a sequence of operational questions rather than merely
cataloguing columns:

1. **Can the data be joined safely?** Yes - use card ID and keep card moves
   separate from card identity.
2. **Can the deck initialize?** Yes - the list has 60 recognized cards and
   no non-Basic-Energy entry above four copies.
3. **Can it set up consistently?** This is the principal weakness: only six
   Basic Pokemon create substantial opening-hand mulligan risk.
4. **Are evolutions structurally live?** Yes - the Stage 1 line has its
   required previous stage in the deck.
5. **Can printed damage define the policy alone?** No - variable damage and
   effects require simulator-backed episode evaluation.

**Next controlled experiment.** Keep the promoted development-first policy
fixed and test one immediate-knockout exception. Do not change deck
construction in the same experiment. After that policy result is isolated,
test Basic-Pokemon count as a separate deck-consistency intervention.

**Limitation.** Printed card data does not reveal realized draw order,
strategic synergy, or opponent prevalence. Those require controlled episode
evidence with named-context telemetry.